# Credit Card Fraud Detection — Reproducible ML Workflow

**Internship project | Elevate Labs**

This notebook provides an end-to-end experiment for the Kaggle Credit Card Fraud Detection dataset. It includes data inspection, safe train/test preprocessing, Logistic Regression, XGBoost, Isolation Forest, Local Outlier Factor (LOF), and model evaluation plots.

> Run cells from top to bottom. This notebook trains a **new experimental model**. Results may differ from the model currently used by the Streamlit app. The notebook saves experimental artifacts under `models/experiment/` and does not overwrite the existing root-level model files.


## 1. Imports and paths

In [ ]:
from pathlib import Path
import json
import warnings
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.metrics import (
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    precision_score, recall_score, f1_score, roc_auc_score,
    average_precision_score, RocCurveDisplay, PrecisionRecallDisplay
)
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")
RANDOM_STATE = 42

DATA_PATH = Path("data/creditcard.csv")
if not DATA_PATH.exists():
    DATA_PATH = Path("../data/creditcard.csv")
if not DATA_PATH.exists():
    raise FileNotFoundError("Download creditcard.csv from Kaggle and place it in data/.")

FIG_DIR = Path("reports/figures")
if not FIG_DIR.exists() and Path("../reports").exists():
    FIG_DIR = Path("../reports/figures")
FIG_DIR.mkdir(parents=True, exist_ok=True)

MODEL_DIR = Path("models/experiment")
MODEL_DIR.mkdir(parents=True, exist_ok=True)

print("Dataset:", DATA_PATH.resolve())
print("Figures:", FIG_DIR.resolve())
print("Experimental artifacts:", MODEL_DIR.resolve())

## 2. Load and inspect the dataset

In [ ]:
df = pd.read_csv(DATA_PATH)
print("Dataset shape:", df.shape)
display(df.head())
display(df.dtypes)
print("Missing values:", int(df.isna().sum().sum()))
print("Duplicate rows:", int(df.duplicated().sum()))
if "Class" not in df.columns:
    raise ValueError("Expected target column 'Class' was not found.")
display(df["Class"].value_counts().rename(index={0: "Legitimate", 1: "Fraud"}))
display((df["Class"].value_counts(normalize=True) * 100).round(4))

In [ ]:
plt.figure(figsize=(6, 4))
sns.countplot(data=df, x="Class")
plt.title("Class Distribution (0 = Legitimate, 1 = Fraud)")
plt.xlabel("Transaction class")
plt.ylabel("Number of transactions")
plt.tight_layout()
plt.savefig(FIG_DIR / "class_distribution.png", dpi=180)
plt.show()

## 3. Prepare features and split data

The label is `Class`. The feature columns should include `Time`, `Amount`, and PCA components `V1`–`V28`. The split is stratified so that the fraud proportion is represented in both subsets. The scaler is fitted only on the training partition to prevent test-data leakage.

In [ ]:
target = "Class"
feature_columns = [col for col in df.columns if col != target]
X = df[feature_columns].copy()
y = df[target].astype(int).copy()

required = {"Time", "Amount"}
if not required.issubset(X.columns):
    raise ValueError(f"Missing required features: {required - set(X.columns)}")
if y.nunique() != 2:
    raise ValueError("Expected a binary classification target.")

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE
)

scaler = StandardScaler()
scale_columns = ["Time", "Amount"]
X_train = X_train_raw.copy()
X_test = X_test_raw.copy()
X_train.loc[:, scale_columns] = scaler.fit_transform(X_train_raw[scale_columns])
X_test.loc[:, scale_columns] = scaler.transform(X_test_raw[scale_columns])

print("Training rows:", len(X_train), "| Test rows:", len(X_test))
print("Training fraud rate:", round(float(y_train.mean()), 6))
print("Test fraud rate:", round(float(y_test.mean()), 6))

## 4. Evaluation helper

In [ ]:
def evaluate_model(name, y_true, y_pred, y_score=None):
    print(f"\\n===== {name} =====")
    print(classification_report(y_true, y_pred, digits=4, zero_division=0))
    cm = confusion_matrix(y_true, y_pred)
    print("Confusion matrix [[TN, FP], [FN, TP]]:")
    print(cm)
    result = {
        "Model": name,
        "Precision (fraud)": precision_score(y_true, y_pred, zero_division=0),
        "Recall (fraud)": recall_score(y_true, y_pred, zero_division=0),
        "F1 (fraud)": f1_score(y_true, y_pred, zero_division=0)
    }
    if y_score is not None:
        result["ROC-AUC"] = roc_auc_score(y_true, y_score)
        result["Average Precision"] = average_precision_score(y_true, y_score)
    return result, cm

## 5. Logistic Regression baseline

Class weighting is used to make the baseline more attentive to the minority fraud class. The model is trained only on the training partition.

In [ ]:
log_reg = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    solver="liblinear",
    random_state=RANDOM_STATE
)
log_reg.fit(X_train, y_train)
lr_pred = log_reg.predict(X_test)
lr_score = log_reg.predict_proba(X_test)[:, 1]
lr_result, lr_cm = evaluate_model("Logistic Regression", y_test, lr_pred, lr_score)

## 6. XGBoost classifier

`scale_pos_weight` is set from the training class counts. This is a class-imbalance parameter; it is calculated using only training labels.

In [ ]:
negative_count = int((y_train == 0).sum())
positive_count = int((y_train == 1).sum())
scale_pos_weight = negative_count / max(positive_count, 1)

xgb_model = XGBClassifier(
    n_estimators=350,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.9,
    colsample_bytree=0.9,
    objective="binary:logistic",
    eval_metric="logloss",
    scale_pos_weight=scale_pos_weight,
    random_state=RANDOM_STATE,
    n_jobs=-1
)
xgb_model.fit(X_train, y_train)
xgb_pred = xgb_model.predict(X_test)
xgb_score = xgb_model.predict_proba(X_test)[:, 1]
xgb_result, xgb_cm = evaluate_model("XGBoost", y_test, xgb_pred, xgb_score)

## 7. Isolation Forest anomaly-detection baseline

Isolation Forest is unsupervised: it does not use `y_train` to learn fraud patterns. The training fraud proportion is used only as an approximate contamination setting. A subset is used to keep runtime manageable.

In [ ]:
contamination = float(y_train.mean())
fit_n = min(50000, len(X_train))
fit_idx = X_train.sample(n=fit_n, random_state=RANDOM_STATE).index
iso = IsolationForest(
    n_estimators=200,
    contamination=contamination,
    random_state=RANDOM_STATE,
    n_jobs=-1
)
iso.fit(X_train.loc[fit_idx])
iso_pred = (iso.predict(X_test) == -1).astype(int)
iso_result, iso_cm = evaluate_model("Isolation Forest", y_test, iso_pred)

## 8. Local Outlier Factor (LOF) anomaly-detection baseline

LOF is configured with `novelty=True` so it can score unseen test observations after fitting. A capped training sample is used because LOF can be expensive on large datasets.

In [ ]:
lof_n = min(20000, len(X_train))
lof_idx = X_train.sample(n=lof_n, random_state=RANDOM_STATE).index
lof = LocalOutlierFactor(
    n_neighbors=20,
    contamination=contamination,
    novelty=True,
    n_jobs=-1
)
lof.fit(X_train.loc[lof_idx])
lof_pred = (lof.predict(X_test) == -1).astype(int)
lof_result, lof_cm = evaluate_model("Local Outlier Factor", y_test, lof_pred)

## 9. Compare all four approaches

In [ ]:
comparison = pd.DataFrame([lr_result, xgb_result, iso_result, lof_result])
display(comparison.round(4))
comparison.to_csv(Path("reports/model_comparison.csv") if Path("reports").exists()
                  else Path("../reports/model_comparison.csv"), index=False)

ax = comparison.set_index("Model")[[
    "Precision (fraud)", "Recall (fraud)", "F1 (fraud)"
]].plot(kind="bar", figsize=(10, 5), ylim=(0, 1), rot=15)
ax.set_title("Model Comparison — Fraud Class")
ax.set_ylabel("Score")
plt.tight_layout()
plt.savefig(FIG_DIR / "model_comparison.png", dpi=180)
plt.show()

## 10. XGBoost confusion matrix, ROC curve, and precision–recall curve

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay.from_predictions(
    y_test, xgb_pred,
    display_labels=["Legitimate", "Fraud"],
    values_format="d",
    cmap="Blues",
    ax=ax
)
ax.set_title("XGBoost Confusion Matrix")
plt.tight_layout()
plt.savefig(FIG_DIR / "xgboost_confusion_matrix.png", dpi=180)
plt.show()

fig, ax = plt.subplots(figsize=(6, 5))
RocCurveDisplay.from_predictions(y_test, xgb_score, name="XGBoost", ax=ax)
ax.set_title("XGBoost ROC Curve")
plt.tight_layout()
plt.savefig(FIG_DIR / "xgboost_roc_curve.png", dpi=180)
plt.show()

fig, ax = plt.subplots(figsize=(6, 5))
PrecisionRecallDisplay.from_predictions(y_test, xgb_score, name="XGBoost", ax=ax)
ax.set_title("XGBoost Precision–Recall Curve")
plt.tight_layout()
plt.savefig(FIG_DIR / "xgboost_precision_recall_curve.png", dpi=180)
plt.show()

## 11. Save experimental model artifacts and summary

The artifacts are written to `models/experiment/` so that your existing deployed model files in the repository root are not overwritten.

In [ ]:
joblib.dump(scaler, MODEL_DIR / "feature_scaler.pkl")
joblib.dump(feature_columns, MODEL_DIR / "feature_columns.pkl")
xgb_model.save_model(str(MODEL_DIR / "xgboost_fraud_model.json"))
joblib.dump(xgb_model, MODEL_DIR / "xgboost_fraud_model.pkl")

summary = {
    "dataset_rows": int(len(df)),
    "feature_count": int(len(feature_columns)),
    "train_rows": int(len(X_train)),
    "test_rows": int(len(X_test)),
    "fraud_rate": float(y.mean()),
    "xgboost_metrics": xgb_result,
    "xgboost_confusion_matrix": xgb_cm.tolist()
}
with open(MODEL_DIR / "evaluation_summary.json", "w") as f:
    json.dump(summary, f, indent=2)
print(json.dumps(summary, indent=2))

## 12. Final checks and interpretation

1. Confirm the notebook ran from the first cell to the last without errors.
2. Inspect the saved class-distribution and evaluation plots.
3. Record the metrics produced by **your own run**; do not assume they will reproduce previously reported values.
4. Keep the existing root-level model artifacts until the new model is validated against the Streamlit app.
5. The raw dataset is excluded from Git by `.gitignore`; follow the dataset provider's terms.

**Metric reminder:** precision answers “of the transactions flagged as fraud, how many were fraud?” Recall answers “of all actual fraud cases, how many did the model catch?” False positives and false negatives have different operational costs. This is an academic prototype, not a production banking decision system.
